In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 11.5 MB/s eta 0:00:00


In [ ]:
import os
import cv2
import torch
from ultralytics import YOLO
import xml.etree.ElementTree as ET
from xml.dom import minidom
from tqdm import tqdm

INPUT_DIR = "/content/drive/MyDrive/CapstoneProject/make/data/batch_42"
OUTPUT_XML_PATH = "/content/drive/MyDrive/CapstoneProject/make/data/batch_42_annotations.xml"

VEHICLE_CLASSES = [2, 3, 5, 7]

TARGET_LABEL = "lane_keeping"

def create_cvat_xml(input_dir, output_path):
    model = YOLO('/content/drive/MyDrive/CapstoneProject/models/model(26n_ver1)/yolo/weights/best.pt')

    root = ET.Element("annotations")

    image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    image_files.sort()

    print(f"Tìm thấy {len(image_files)} ảnh. Bắt đầu xử lý...")

    for i, filename in enumerate(tqdm(image_files)):
        img_path = os.path.join(input_dir, filename)

        img = cv2.imread(img_path)
        if img is None:
            continue
        height, width = img.shape[:2]

        results = model.predict(img_path, conf=0.4, verbose=False)
        result = results[0]

        image_node = ET.SubElement(root, "image", id=str(i), name=filename, width=str(width), height=str(height))

        boxes = result.boxes
        for box in boxes:
            cls_id = int(box.cls[0])

            if cls_id in VEHICLE_CLASSES:
                x1, y1, x2, y2 = box.xyxy[0].tolist()

                ET.SubElement(image_node, "box",
                              label=TARGET_LABEL,
                              occluded="0",
                              source="manual",
                              xtl=f"{x1:.2f}",
                              ytl=f"{y1:.2f}",
                              xbr=f"{x2:.2f}",
                              ybr=f"{y2:.2f}",
                              z_order="0")

    xml_str = minidom.parseString(ET.tostring(root)).toprettyxml(indent="  ")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(xml_str)

    print(f"\nĐã hoàn thành! File annotation được lưu tại: {output_path}")

if __name__ == "__main__":
    if os.path.exists(INPUT_DIR):
        create_cvat_xml(INPUT_DIR, OUTPUT_XML_PATH)
    else:
        print("Không tìm thấy đường dẫn thư mục ảnh!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Tìm thấy 1231 ảnh. Bắt đầu xử lý...


100%|██████████| 1231/1231 [06:56<00:00,  2.95it/s]



Đã hoàn thành! File annotation được lưu tại: /content/drive/MyDrive/CapstoneProject/make/data/batch_42_annotations.xml
